# Question 2: Resampling and Frequency Conversion

This question focuses on resampling operations and frequency conversion using ICU monitoring data (hourly) and patient vital signs data (daily).

## Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('default')
sns.set_style('whitegrid')

# Create output directory
os.makedirs('output', exist_ok=True)

## Part 2.1: Load and Prepare Data

**Note:** These datasets have realistic characteristics:
- **ICU Monitoring**: 75 patients with variable stay lengths (2-30 days). Not all patients are present for the entire 6-month period - patients are admitted and discharged at different times.
- **Patient Vitals**: Already contains some missing visits (~5% missing data). This is realistic and will be useful for practicing missing data handling.

In [3]:
# Load ICU monitoring data (hourly)
icu_monitoring = pd.read_csv('data/icu_monitoring.csv')

# Load patient vitals data (daily) - for comparison
patient_vitals = pd.read_csv('data/patient_vitals.csv')

print("ICU monitoring shape:", icu_monitoring.shape)
print("Patient vitals shape:", patient_vitals.shape)

# Convert datetime columns and set as index
icu_monitoring['datetime'] = pd.to_datetime(icu_monitoring['datetime'])
icu_monitoring = icu_monitoring.set_index('datetime')

patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])
patient_vitals = patient_vitals.set_index('date')

print("\nICU monitoring sample:")
print(icu_monitoring.head())
print("\nPatient vitals sample:")
print(patient_vitals.head())

# Check data characteristics
print(f"\nICU patients: {icu_monitoring['patient_id'].nunique()}")
print(f"ICU date range: {icu_monitoring.index.min()} to {icu_monitoring.index.max()}")
print(f"\nPatient vitals patients: {patient_vitals['patient_id'].nunique()}")
print(f"Patient vitals date range: {patient_vitals.index.min()} to {patient_vitals.index.max()}")

ICU monitoring shape: (86400, 7)
Patient vitals shape: (18250, 7)

ICU monitoring sample:
                    patient_id  heart_rate  blood_pressure_systolic  \
datetime                                                              
2023-01-01 00:00:00     ICU001   82.000000                      126   
2023-01-01 01:00:00     ICU001   98.294095                      128   
2023-01-01 02:00:00     ICU001  103.500000                      129   
2023-01-01 03:00:00     ICU001   91.535534                      136   
2023-01-01 04:00:00     ICU001   87.330127                      129   

                     blood_pressure_diastolic  oxygen_saturation  temperature  
datetime                                                                       
2023-01-01 00:00:00                        65                 96    98.783988  
2023-01-01 01:00:00                        67                 95    99.186212  
2023-01-01 02:00:00                        68                 94    98.800638  
2023-01-01 0

## Part 2.2: Time Series Selection

**⚠️ WARNING: Sort Index Before Date Selection!**
Since multiple patients share the same date, the `patient_vitals` index is non-monotonic (not strictly increasing). **You MUST sort the index first** before using `.loc` with date ranges:

In [4]:
patient_vitals = patient_vitals.sort_index()
print("patient_vitals sorted!!!")

patient_vitals sorted!!!


Without sorting, pandas cannot reliably handle date range selections and may return unexpected results or errors.

**TODO: Perform time series indexing and selection**

In [5]:
# TODO: Select data by specific dates
# Note: Not all patients may have data on January 1, 2023 (some start later)
print("1. Selcting data by specific dates")
# Important: Sort the index first since multiple patients share the same date
# patient_vitals = patient_vitals.sort_index()  # Sort for reliable date-based selection
patient_vitals = patient_vitals.sort_index()
# january_first = None  # Select January 1, 2023 from patient_vitals
january_first = patient_vitals.loc['2023-01-01']
# print("January 1, 2023 data:", january_first)
print("January 1, 2023 data:", january_first)
#print(f"Records on Jan 1: {len(january_first)} (some patients may start later)")
print(f"Records on Jan 1: {len(january_first)} (some patients may start later)")
print(f" Unique patients: {january_first['patient_id'].nunique()}")
print()

# TODO: Select data by date ranges
# january_data = None  # Select entire January 2023
january_data = patient_vitals.loc['2023-01']
# print("January 2023 shape:", january_data.shape)
print("2. January 2023 shape:", january_data.shape)
print()

# TODO: Select data by time periods
print("3. Select data by time periods (first quarter, entire year)")
# first_quarter = None  # Select Q1 2023
first_quarter = patient_vitals.loc['2023-Q1']
print(f"First quarter 2023 (Q1):")
print(f" Shape: {first_quarter.shape}")
print()
# entire_year = None  # Select all of 2023 (will include patients with partial year data)
entire_year = patient_vitals.loc['2023']
print(f"Entire year 2023:")
print(f" Shape: {entire_year.shape}")
print()

# TODO: Select first and last periods using .loc
print("4. Selecting first and last periods")
# first_week = patient_vitals.loc[:patient_vitals.index.min() + pd.Timedelta(days=6)]  # First 7 days
first_week = patient_vitals.loc[:patient_vitals.index.min() + pd.Timedelta(days=6)]
print(f"First week of data:")
print(f" Shape: {first_week.shape}")
print()
# last_week = patient_vitals.loc[patient_vitals.index.max() - pd.Timedelta(days=6):]  # Last 7 days
last_week = patient_vitals.loc[patient_vitals.index.max() - pd.Timedelta(days=6):]
print(f"Last week of data:")
print(f" Shape: {last_week.shape}")
print()

# TODO: Use truncate() method
print("5. Use truncate() method")
# Note: truncate() requires a sorted index. Sort first if needed: patient_vitals = patient_vitals.sort_index()
# data_after_june = None  # Truncate before June 1, 2023
data_after_june = patient_vitals.truncate(before = '2023-06-01')
print(f"Data after June 1, 2023:")
print(f" Shape: {data_after_june.shape}")
print()
# data_before_september = None  # Truncate after August 31, 2023
data_before_september = patient_vitals.truncate(after = '2023-08-31')
print(f"Data before September 1, 2023:")
print(f" Shape: {data_before_september.shape}")
print()

# TODO: Use selected data for analysis
# Compare average temperature between first quarter and data after June
print("6. Temperature Analysis")
# print(f"\nFirst quarter average temperature: {first_quarter['temperature'].mean():.2f}°F")
print(f"\nFirst quarter average temperature: {first_quarter['temperature'].mean():.2f}°F")
# print(f"After June average temperature: {data_after_june['temperature'].mean():.2f}°F")
print(f"After June average temperature: {data_after_june['temperature'].mean():.2f}°F")
# print(f"First week average temperature: {first_week['temperature'].mean():.2f}°F")
print(f"First week average temperature: {first_week['temperature'].mean():.2f}°F")
# print(f"Last week average temperature: {last_week['temperature'].mean():.2f}°F")
print(f"Last week average temperature: {last_week['temperature'].mean():.2f}°F")
print()


# TODO: Select business hours (9 AM to 5 PM)
print("7. Business hours (9 AM to 5 PM)")
# business_hours = None  # Use between_time()
business_hours = icu_monitoring.between_time('09:00', '17:00')
# print("Business hours data shape:", business_hours.shape)
print("Business hours data shape:", business_hours.shape)
print()

# TODO: Select specific time (noon readings)
print("8. Select noon readings")
# noon_data = None  # Use at_time('12:00')
noon_data = icu_monitoring.at_time('12:00')
print(f"Noon readings (12:00 PM):")
print(f" Shape: {noon_data.shape}")
print()

# TODO: Use time-based selection for analysis
print("9. Time-based selection for analysis")
# Compare vital signs during business hours vs other times
# all_hours_avg = icu_monitoring.select_dtypes(include=[np.number]).mean()
all_hours_avg = icu_monitoring.select_dtypes(include=[np.number]).mean()
# business_hours_avg = business_hours.select_dtypes(include=[np.number]).mean()
business_hours_avg = business_hours.select_dtypes(include=[np.number]).mean()
# print(f"\nAverage heart rate - All hours: {all_hours_avg['heart_rate']:.1f} bpm")
print(f"\nAverage heart rate - All hours: {all_hours_avg['heart_rate']:.1f} bpm")
print()
# print(f"Average heart rate - Business hours: {business_hours_avg['heart_rate']:.1f} bpm")
print(f"Average heart rate - Business hours: {business_hours_avg['heart_rate']:.1f} bpm")
print()
# print(f"Average temperature - All hours: {all_hours_avg['temperature']:.1f}°F")
print(f"Average temperature - All hours: {all_hours_avg['temperature']:.1f}°F")
print()
# print(f"Average temperature - Business hours: {business_hours_avg['temperature']:.1f}°F")
print(f"Average temperature - Business hours: {business_hours_avg['temperature']:.1f}°F")
print()

1. Selcting data by specific dates
January 1, 2023 data:            patient_id  temperature  heart_rate  blood_pressure_systolic  \
date                                                                      
2023-01-01      P0001    98.389672          71                      119   
2023-01-01      P0024    97.552103          71                      111   
2023-01-01      P0025    98.806201          83                      118   
2023-01-01      P0047    98.943464          63                      110   
2023-01-01      P0026    98.758551          75                      114   
2023-01-01      P0027    98.462467          90                      119   
2023-01-01      P0028    99.632166          71                      117   
2023-01-01      P0029    98.881121          68                      121   
2023-01-01      P0030    98.403614          73                      122   
2023-01-01      P0031    98.410626          63                      113   
2023-01-01      P0046    98.507553         

## Part 2.3: Resampling Operations

**TODO: Perform resampling and frequency conversion**

**Important Note:** When resampling DataFrames that contain non-numeric columns (like `patient_id`), you'll get an error if you try to aggregate them with numeric functions like `mean()`. Use `df.select_dtypes(include=[np.number])` to select only numeric columns before resampling, or specify which columns to aggregate in `.agg()`.

In [7]:
# TODO: Resample hourly ICU data to daily
print("1. Resampling hourly ICU data to daily")
# Note: Exclude non-numeric columns like 'patient_id' when resampling
# Select only numeric columns before resampling
# numeric_cols = icu_monitoring.select_dtypes(include=[np.number]).columns
numeric_cols = icu_monitoring.select_dtypes(include=[np.number]).columns
# icu_daily = icu_monitoring[numeric_cols].resample('D').mean()
icu_daily = icu_monitoring[numeric_cols].resample('D').mean()
# print("ICU daily shape:", icu_daily.shape)
print("ICU daily shape:", icu_daily.shape)
print(icu_daily.head())
print()

# TODO: Resample daily patient data to weekly
print("2. Resampling daily ICU data to weekly")
# Note: Exclude 'patient_id' column when resampling
# Select only numeric columns before resampling
# numeric_cols_pv = patient_vitals.select_dtypes(include=[np.number]).columns
numeric_cols_pv = patient_vitals.select_dtypes(include=[np.number]).columns
# patient_vitals_weekly = patient_vitals[numeric_cols_pv].resample('W').mean()
patient_vitals_weekly = patient_vitals[numeric_cols_pv].resample('W').mean()
# print("Weekly resampled shape:", patient_vitals_weekly.shape)
print("Weekly resampled shape:", patient_vitals_weekly.shape)
print(patient_vitals_weekly.head())
print()

# TODO: Resample daily patient data to monthly
print("2. Resampling daily ICU data to monthly")
# patient_vitals_monthly = None  # Resample to monthly with mean aggregation (use freq='ME' for Month End)
patient_vitals_monthly = patient_vitals[numeric_cols_pv].resample('ME').mean()
# print("Monthly resampled shape:", patient_vitals_monthly.shape)
print("Monthly resampled shape:", patient_vitals_monthly.shape)
print()

# TODO: Use different aggregation functions (mean, sum, max, min)
print("3. Summary aggregation functions (mean, sum, max, min)")
# icu_daily_stats = None  # Resample with multiple aggregations
# Example: resample('D').agg({'heart_rate': ['mean', 'max', 'min'], 
#                             'temperature': 'mean'})
icu_daily_stats = icu_monitoring[numeric_cols].resample('D').agg({
    'heart_rate': ['mean', 'max', 'min', 'std'],
    'temperature': ['mean', 'max', 'min'],
    'blood_pressure_systolic': ['mean', 'max', 'min'],
    'oxygen_saturation': 'mean'
})
print(f"\nICU daily statistics shape: {icu_daily_stats.shape}")
print(icu_daily_stats.head())
print()

# TODO: Handle missing values during resampling
print("4. Handling missing values during resampling")
# Demonstrate upsampling (monthly to daily) creates missing values
# Note: When upsampling, use .asfreq() to create missing values, or use .resample() with aggregation
# monthly_to_daily = None  # Upsample monthly data to daily (use .asfreq() or .resample('D'))
monthly_to_daily = patient_vitals_monthly.resample('D').asfreq()
print(f"Original monthly data: {patient_vitals_monthly.shape}")
print(f"Upsampled to daily: {monthly_to_daily.shape}")
# print("Missing values after upsampling:", monthly_to_daily.isna().sum())
print("Missing values after upsampling:", monthly_to_daily.isna().sum())
print(monthly_to_daily.head())
print()

# TODO: Compare different resampling frequencies
# Create a DataFrame comparing resampling results at different frequencies
# Important: Since patient_vitals contains multiple patients per date, you need to aggregate by date first
# to create a single daily time series for comparison.
# Why aggregation is needed: The patient_vitals DataFrame has multiple rows per date (one for each patient),
# so we need to average across patients for each date to create a single daily time series that can be
# meaningfully compared with the weekly and monthly resampled data. Without aggregation, resampling would
# operate on each patient's time series separately, making it difficult to compare frequencies meaningfully.
print("6. Comparing different resampling frequencies")
print()
# Steps:
# 1. Since 'date' is currently the index, reset it to a column first, then aggregate by date
#    Note: groupby('date').mean() automatically sets 'date' as the index in the result, so you don't need
#    to call set_index('date') again after groupby.
#    patient_vitals_reset = patient_vitals[numeric_cols_pv].reset_index()
#    patient_vitals_daily_agg = patient_vitals_reset.groupby('date').mean()
#    # The date is already the index after groupby, so no need to set_index again
print("6.1 Resetting 'date' index as column for groupby")
patient_vitals_reset = patient_vitals[numeric_cols_pv].reset_index()
# Group by date and find the mean across all patients, groupby sets 'date' as index automatically
patient_vitals_daily_agg = patient_vitals_reset.groupby('date').mean()
print(f"Original data (multi-patient): {patient_vitals.shape}")
print(f"Aggregated by date (single series): {patient_vitals_daily_agg.shape}")
print(f"\nSample of aggregated daily data:")
print(patient_vitals_daily_agg.head())
print()
# 2. Compare the aggregated daily data with weekly and monthly resampled data
# Use patient_vitals data resampled to different frequencies:
# - Original daily data (aggregated by date): patient_vitals_daily_agg
# - Weekly resampled (patient_vitals_weekly) 
# - Monthly resampled (patient_vitals_monthly)
# Include columns: frequency, date_range, row_count, mean_temperature, std_temperature
# Use the 'temperature' column from each resampled dataset
# Example structure:
# resampling_comparison = pd.DataFrame({
#     'frequency': ['daily', 'weekly', 'monthly'],
#     'date_range': [...],  # Use index.min() and index.max() for each dataset
#     'row_count': [...],  # Use len() for each dataset
#     'mean_temperature': [...],  # Use .mean() on 'temperature' column for each dataset
#     'std_temperature': [...]   # Use .std() on 'temperature' column for each dataset
# })
print("6.2 Comparing aggregated daily data with wekly and monthly resampled data")
resampling_comparison = pd.DataFrame({
    'frequency': ['daily', 'weekly', 'monthly'],
    'date_range': [
        f"{patient_vitals_daily_agg.index.min().date()} to {patient_vitals_daily_agg.index.max().date()}",
        f"{patient_vitals_weekly.index.min().date()} to {patient_vitals_weekly.index.max().date()}",
        f"{patient_vitals_monthly.index.min().date()} to {patient_vitals_monthly.index.max().date()}"
    ],
    'row_count': [
        len(patient_vitals_daily_agg),
        len(patient_vitals_weekly),
        len(patient_vitals_monthly)
    ],
    'mean_temperature': [
        patient_vitals_daily_agg['temperature'].mean(),
        patient_vitals_weekly['temperature'].mean(),
        patient_vitals_monthly['temperature'].mean()
    ],
    'std_temperature': [
        patient_vitals_daily_agg['temperature'].std(),
        patient_vitals_weekly['temperature'].std(),
        patient_vitals_monthly['temperature'].std()
    ]
})
print(resampling_comparison.to_string(index=False))
print()

# TODO: Save results as 'output/q2_resampling_analysis.csv'
# resampling_comparison.to_csv('output/q2_resampling_analysis.csv', index=False)
resampling_comparison.to_csv('output/q2_resampling_analysis.csv', index=False)
print("Saved: output/q2_resampling_analysis.csv")

1. Resampling hourly ICU data to daily
ICU daily shape: (180, 5)
            heart_rate  blood_pressure_systolic  blood_pressure_diastolic  \
datetime                                                                    
2023-01-01   81.793729               119.729167                 72.481250   
2023-01-02   81.479854               119.714583                 72.400000   
2023-01-03   81.767332               119.893750                 72.568750   
2023-01-04   81.852771               119.583333                 72.329167   
2023-01-05   81.730187               119.558333                 72.293750   

            oxygen_saturation  temperature  
datetime                                    
2023-01-01          96.366667    98.528348  
2023-01-02          96.364583    98.498305  
2023-01-03          96.381250    98.534337  
2023-01-04          96.385417    98.542113  
2023-01-05          96.410417    98.536312  

2. Resampling daily ICU data to weekly
Weekly resampled shape: (53, 5)
        

## Part 2.4: Missing Data Handling

**💡 TIP: High Percentage of Missing Data is Expected!**
When upsampling from monthly to daily frequency, you'll create approximately 96% missing data (only 12 month-end dates have values out of 365 days). This is normal and expected for upsampling - don't be alarmed!

**Approach:** Create missing values by upsampling monthly data to daily frequency. This creates a clear, structured pattern of missing data that's ideal for practicing imputation methods.

**TODO: Handle missing data in time series**

In [ ]:
# TODO: Identify missing values in time series
# Use the monthly resampled data from Part 2.3 and upsample to daily:
#   - Take patient_vitals_monthly['temperature']
#   - Upsample to daily frequency using .resample('D').asfreq()
#   - This creates missing values for all days except month-end dates (~96% missing)
print("1. Upsampling montly data to daily frequency to create missing values for all days except month-end dates")
print()
# ts_with_missing = None  # Time series with missing values
ts_with_missing = patient_vitals_monthly['temperature'].resample('D').asfreq()
# print("Missing value count:", ts_with_missing.isna().sum())
print("Missing value count:", ts_with_missing.isna().sum())
# print("Missing value percentage:", ts_with_missing.isna().sum() / len(ts_with_missing) * 100)
print("Missing value percentage:", ts_with_missing.isna().sum() / len(ts_with_missing) * 100)
print(ts_with_missing.head())

# TODO: Use forward fill and backward fill
print("2. Forward and backward fill")
# ts_ffill = None  # Forward fill missing values (use .ffill() method)
ts_ffill = ts_with_missing.ffill()
print("Forward Fill (ffill):")
print(f"  Missing values remaining: {ts_ffill.isna().sum()}")
print(f"  Method: Fills last known value forward to fill gaps")
print()
# ts_bfill = None  # Backward fill missing values (use .bfill() method)
ts_bfill = ts_with_missing.bfill()
print("Backward Fill (bfill):")
print(f"  Missing values remaining: {ts_bfill.isna().sum()}")
print(f"  Method: Fills next known value backward to fill gaps")
print()

# TODO: Use interpolation methods
print("3. Interpolation")
# ts_interpolated = None  # Interpolate missing values
# ts_interpolated_linear = None  # Linear interpolation
ts_interpolated_linear = ts_with_missing.interpolate(method='linear')
print("Linear Interpolation:")
print(f"  Missing values remaining: {ts_interpolated_linear.isna().sum()}")
print(f"  Linear interpolation: Straight line between known points")
print()
# ts_interpolated_time = None  # Time-based interpolation
ts_interpolated_time = ts_with_missing.interpolate(method='time')
print("Time-based Interpolation:")
print(f"  Missing values remaining: {ts_interpolated_time.isna().sum()}")
print(f"  Time-based interpolation: Interpolates based on differences between date data points")
print()

# TODO: Use rolling mean for imputation
print("4. Interpolation using rolling mean")
# ts_rolling_imputed = None  # Fill missing with rolling mean
rolling_mean = ts_with_missing.rolling(window=7, center=True, min_periods=1).mean()
    # define 7-day rolling mean first
ts_rolling_imputed = ts_with_missing.fillna(rolling_mean)
print(f"Imputation with rolling mean:")
print(f"  Window size: 7 days")
print(f"  Missing values remaining: {ts_rolling_imputed.isna().sum()}")
print(f"  Method: Fill with average of surrounding 7-day window")
print()

# TODO: Create missing data report
# Document your missing data handling with the following sections:
# 1. Missing value summary: Total count and percentage
# 2. Missing data patterns: When/why data is missing (by month, day of week, etc.)
# 3. Imputation method: Which method you used (forward fill, backward fill, interpolation, rolling mean)
# 4. Rationale: Why you chose that method
# 5. Pros and cons: Advantages and limitations of your approach
# 6. Example: Show at least one example of missing data before and after imputation
# Minimum length: 300 words

# Calculating missing data 
missing_count = ts_with_missing.isna().sum()
total_count = len(ts_with_missing)
missing_percentage = (missing_count / total_count) * 100

# data.frame comparing differnt interpolation methods in the first 10 days of January 2023
start_date = ts_with_missing.index.min()
end_date = start_date + pd.Timedelta(days=9)
interp_comparison = pd.DataFrame({
    'Original': ts_with_missing.loc[start_date:end_date],
    'Forward_Fill': ts_ffill.loc[start_date:end_date],
    'Backward_Fill': ts_bfill.loc[start_date:end_date],
    'Linear': ts_interpolated_linear.loc[start_date:end_date],
    'Time': ts_interpolated_time.loc[start_date:end_date],
})

missing_data_report = f"""
Missing Data Handling Report

TODO: Document your missing data handling:
- How many missing values did you find?
We can see from the dataset:
Total Observations: {total_count} days
Missing Values: {missing_count} observations

- What percentage of data was missing?
Missing percentage: {missing_percentage:.1f}%

- Which method did you use to fill missing values?
I used linear interpolation as the method to fill missing values.
ts_interpolated_linear = ts_with_missing.interpolate(method='linear')

- Why did you choose that method?
The linear interpolation method draws a straight line between consecutive known data points, which is a good strategy to estimate missing values that lie between the two timepoints. It is highly likely that missing data will fall at or around this line of best fit if the data follows a predictable trend. 

- What are the pros/cons of your approach?
Linear interpolation is a good strategy to esimate this missing data, since we upsampled monthly observations to daily. This provides regular, intervalled timepoints between which we need to estimate values. We are looking at changes in patient vitals, which change gradually, steadily, and relatively predictably over time. Linear interpolation is a good fit, since we don't expect extreme outliers to happen in between timepoints. This strategy also will cleanly display trends over time. Other strategies like forward or backward fill may mask changes between timepoints since they assume the data is the same as the previous or next timepoint value. 
That said, linear interpolation assumes that trends change constantly or follow a relatively linear pattern. There may be large fluctuations in the data that this model will not capture. Also, if the endpooints of the time analyses are chosen poorly (i.e. a starting period of stability and an ending point of stability), if an unexpected change happens between the selected timepoints, it will not be captured in the model. 

- Include examples showing missing data patterns
We can look at the first 10 days in January 2023:
{interp_comparison.to_string()}

We can see that: 
* Forward fill keeps the first observed value (98.78˚F) until a new value is input
* Backward fill uses the next observed value (99.00˚F) for all previous days
* Linear interpolation imputes gradual transitions of measurements between known points

For example on February 5, 2023:
* Original: NaN (missing)
* Forward Fill: 98.78°F (copies from previous month-end)
* Backward Fill: 99.00°F (copies from next month-end)
* Linear Interpolation: 98.82°F (halfway between endpoints)

"""


# TODO: Document missing data patterns
# Analyze when/why data is missing
# missing_by_month = ts_with_missing.groupby(ts_with_missing.index.month).apply(lambda x: x.isna().sum())
missing_by_month = ts_with_missing.groupby(ts_with_missing.index.month).apply(lambda x: x.isna().sum())
# missing_by_day = ts_with_missing.groupby(ts_with_missing.index.dayofweek).apply(lambda x: x.isna().sum())
missing_by_day = ts_with_missing.groupby(ts_with_missing.index.dayofweek).apply(lambda x: x.isna().sum())
# missing_patterns = f"Missing by month:\n{missing_by_month}\n\nMissing by day of week:\n{missing_by_day}"
missing_patterns = f"Missing by month:\n{missing_by_month}\n\nMissing by day of week:\n{missing_by_day}"

# TODO: Save results as 'output/q2_missing_data_report.txt'
# with open('output/q2_missing_data_report.txt', 'w') as f:
#     f.write(missing_data_report)
#     f.write(f"\n\nMissing patterns:\n{missing_patterns}")
with open('output/q2_missing_data_report.txt', 'w') as f:
    f.write(missing_data_report)
    f.write(f"\n\nMissing patterns:\n{missing_patterns}")

print("Saved: output/q2_missing_data_report.txt")

1. Upsampling montly data to daily frequency to create missing values for all days except month-end dates

Missing value count: 323
Missing value percentage: 96.41791044776119
date
2023-01-31    98.777
2023-02-01       NaN
2023-02-02       NaN
2023-02-03       NaN
2023-02-04       NaN
Freq: D, Name: temperature, dtype: float64
2. Forward and backward fill
Forward Fill (ffill):
  Missing values remaining: 0
  Method: Fills last known value forward to fill gaps

Backward Fill (bfill):
  Missing values remaining: 0
  Method: Fills next known value backward to fill gaps

3. Interpolation
Linear Interpolation:
  Missing values remaining: 0
  Linear interpolation: Straight line between known points

Time-based Interpolation:
  Missing values remaining: 0
  Time-based interpolation: Interpolates based on differences between date data points

4. Interpolation using rolling mean
Imputation with rolling mean:
  Window size: 7 days
  Missing values remaining: 257
  Method: Fill with average of su

## Submission Checklist

Before moving to Question 3, verify you've created:

- [ ] `output/q2_resampling_analysis.csv` - resampling analysis results
- [ ] `output/q2_missing_data_report.txt` - missing data handling report
